# 8 Ablation Benchmark

Train the SNN for each ablation variant under the same protocol; produce the component-ablation table.

Paths are imported from `config.py` at the repository root.
Run the notebooks in numbered order; see the repository README.

In [ ]:
# --- repository configuration -------------------------------------
# All paths come from config.py at the repository root. Edit that
# file once; do not hardcode paths in this notebook.
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
import config
# ------------------------------------------------------------------

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Imports + Configuration                                       ║
# ║                                                                          ║
# ║  AMSTE ABLATION STUDY — SNN training and evaluation                     ║
# ║                                                                          ║
# ║  This notebook trains an identical SNN architecture on each of the 6    ║
# ║  AMSTE ablation variants produced by notebook 3.0, then reports         ║
# ║  classification metrics (F1, Recall, FNR, FPR, AUPRC, AUC, MCC) and     ║
# ║  efficiency metrics (spike rates, SynOps, energy, inference time)       ║
# ║  per variant. Differences in metrics are attributable to the ablated    ║
# ║  AMSTE component because:                                                ║
# ║    1. Same SNN architecture, optimizer, seeds, folds, and data split   ║
# ║    2. Same train/val/test partitioning across all variants             ║
# ║    3. Same hyperparameters and training budget                          ║
# ║    4. Only the input spike encoding differs                             ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import snntorch as snn
from snntorch import surrogate
import snntorch.functional as SF
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score,
    average_precision_score, balanced_accuracy_score, matthews_corrcoef
)
import matplotlib.pyplot as plt
import copy
import time
import warnings
warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────────
# IMPORTANT: This path must match OUTPUT_ROOT from notebook 3.0.
ENCODED_AMSTE = config.ENCODED_ABLATION_DIR
RESULTS_DIR     = config.RESULTS_DIR
CKPT_DIR        = os.path.join(RESULTS_DIR, 'checkpoints')
TENSORBOARD_DIR = os.path.join(RESULTS_DIR, 'tensorboard')

for d in [RESULTS_DIR, CKPT_DIR, TENSORBOARD_DIR]:
    os.makedirs(d, exist_ok=True)

# ── AMSTE ablation variants ──────────────────────────────────────────────────
# Filesystem directory names (must match notebook 3 output directory names).
VARIANT_NAMES = [
    'Basic',
    'No_MAD',
    'No_Polarity',
    'No_MultiScale',
    'No_Spatial',
    'Full_AMSTE',
]

# Pretty display names for tables and plots
VARIANT_DISPLAY = {
    'Basic':         'Basic',
    'No_MAD':        'No-MAD',
    'No_Polarity':   'No-Polarity',
    'No_MultiScale': 'No-MultiScale',
    'No_Spatial':    'No-Spatial',
    'Full_AMSTE':    'Full AMSTE',
}

# Ablation role for grouping in tables and plots
#   lower_bound = all 4 components removed (Basic)
#   leave_one_out = one component removed (No-X variants)
#   full = proposed full method (Full AMSTE)
VARIANT_TYPE = {
    'Basic':         'lower_bound',
    'No_MAD':        'leave_one_out',
    'No_Polarity':   'leave_one_out',
    'No_MultiScale': 'leave_one_out',
    'No_Spatial':    'leave_one_out',
    'Full_AMSTE':    'full',
}

# Component flags — (MAD, Polarity, MultiScale, SpatialGate)
# Used to print the ablation component matrix.
VARIANT_FLAGS = {
    'Basic':         (False, False, False, False),
    'No_MAD':        (False, True,  True,  True ),
    'No_Polarity':   (True,  False, True,  True ),
    'No_MultiScale': (True,  True,  False, True ),
    'No_Spatial':    (True,  True,  True,  False),
    'Full_AMSTE':    (True,  True,  True,  True ),
}

# Colours — matching notebooks 1 and 2 for consistency across the paper
VARIANT_COLORS = {
    'Basic':         '#9E9E9E',   # gray         — simplest baseline
    'No_MAD':        '#F44336',   # red          — fixed threshold
    'No_Polarity':   '#FF9800',   # orange       — |Δx| only
    'No_MultiScale': '#2196F3',   # blue         — single lag
    'No_Spatial':    '#4CAF50',   # green        — no spatial gate
    'Full_AMSTE':    '#FF5722',   # deep orange  — full proposed method
}


# ── Architecture ──────────────────────────────────────────────────────────────
# n_traces from your DAS data. Inferred at runtime from the first .npy file.
NUM_INPUTS_AMSTE = 366       # n_traces in June_24 data (auto-overridden per variant)
NUM_HIDDEN1      = 128
NUM_HIDDEN2      = 64
NUM_OUTPUTS      = 2

# ── Temporal binning ──────────────────────────────────────────────────────────
# Each .npy is (n_traces, n_samples). After BIN_FACTOR=4 binning:
#   n_samples=2000 → NUM_STEPS=500
BIN_FACTOR  = 4
NUM_STEPS   = 500

# ── LIF neuron ────────────────────────────────────────────────────────────────
BETA1     = 0.85   # layer 1 — fast (detects sharp P-wave transients)
BETA2     = 0.90   # layer 2 — medium
BETA3     = 0.95   # output  — slow (accumulates evidence over 1 s)
THRESHOLD = 1.0
SLOPE     = 25     # fast-sigmoid surrogate gradient steepness

# ── Training ──────────────────────────────────────────────────────────────────
BATCH_SIZE          = 256
NUM_EPOCHS          = 100
LEARNING_RATE       = 2e-3
NUM_FOLDS           = 5
SEEDS               = [42, 123, 456]
LR_FACTOR           = 0.5
LR_PATIENCE         = 5
EARLY_STOP_PATIENCE = 15
BURNIN_EPOCHS       = 20   # min epochs before early-stop is allowed
CORRECT_RATE        = 0.8
INCORRECT_RATE      = 0.2

# ── Dataset split ─────────────────────────────────────────────────────────────
TEST_SIZE  = 0.15
SPLIT_SEED = 42

# ── Energy ────────────────────────────────────────────────────────────────────
ENERGY_PER_SYNOP_PJ = 23.6   # Loihi-2 estimate (pJ per synaptic operation)

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
print(f"AMSTE ablation variants: {[VARIANT_DISPLAY[v] for v in VARIANT_NAMES]}")
print(f"Encoded data root: {ENCODED_AMSTE}")
print(f"Total runs planned: {len(VARIANT_NAMES)} variants × "
      f"{NUM_FOLDS} folds × {len(SEEDS)} seeds = "
      f"{len(VARIANT_NAMES) * NUM_FOLDS * len(SEEDS)} training runs")


# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Dataset Utilities                                             ║
# ╚══════════════════════════════════════════════════════════════════════════╝

def load_labels(encoded_root, dataset_tag=None):
    """Load labels.csv. Returns (file_ids, labels, source_tags)."""
    csv_path = os.path.join(encoded_root, 'labels.csv')
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"labels.csv not found: {csv_path}")
    df       = pd.read_csv(csv_path)
    file_ids = df['file_id'].values
    labels   = df['label'].values.astype(int)
    tags     = np.array([dataset_tag or os.path.basename(encoded_root)] * len(labels))
    print(f"  [{dataset_tag or 'data'}] {len(file_ids)} files: "
          f"{np.sum(labels==1)} event + {np.sum(labels==0)} noise")
    return file_ids, labels, tags


def infer_num_inputs(encoded_root, variant_name):
    """Read one .npy file to infer n_traces (NUM_INPUTS)."""
    enc_dir = os.path.join(encoded_root, variant_name)
    for f in os.listdir(enc_dir):
        if f.endswith('.npy'):
            arr = np.load(os.path.join(enc_dir, f))
            return arr.shape[0]   # n_traces
    raise FileNotFoundError(f"No .npy files in {enc_dir}")


def load_spike_array(encoded_root, variant_name, file_id, bin_factor=BIN_FACTOR):
    """Load + bin one .npy file. Returns float32 (NUM_STEPS, n_traces)."""
    path    = os.path.join(encoded_root, variant_name, f'{file_id}.npy')
    spikes  = np.load(path).astype(np.float32)
    n_tr, n_smp = spikes.shape
    n_bins  = n_smp // bin_factor
    trimmed = spikes[:, :n_bins * bin_factor]
    binned  = trimmed.reshape(n_tr, n_bins, bin_factor).sum(axis=2)
    binned  = (binned > 0).astype(np.float32)
    return binned.T                              # (NUM_STEPS, n_traces)


class DASSpikeDataset(Dataset):
    """
    Lazy-loading dataset for one ablation variant.
    Each item: (spike_tensor (NUM_STEPS, n_inputs), label int).
    """
    def __init__(self, file_ids, labels, encoded_root, variant_name,
                  bin_factor=BIN_FACTOR):
        self.file_ids   = np.array(file_ids)
        self.labels     = np.array(labels)
        self.enc_dir    = os.path.join(encoded_root, variant_name)
        self.bin_factor = bin_factor

    def __len__(self):
        return len(self.file_ids)

    def __getitem__(self, idx):
        path   = os.path.join(self.enc_dir, f'{self.file_ids[idx]}.npy')
        spikes = np.load(path).astype(np.float32)
        n_tr, n_smp = spikes.shape
        n_bins  = n_smp // self.bin_factor
        trimmed = spikes[:, :n_bins * self.bin_factor]
        binned  = trimmed.reshape(n_tr, n_bins, self.bin_factor).sum(axis=2)
        binned  = (binned > 0).astype(np.float32).T   # (NUM_STEPS, n_traces)
        return (torch.tensor(binned, dtype=torch.float32),
                torch.tensor(int(self.labels[idx]), dtype=torch.long))


def available_variants(encoded_root):
    """Return ablation variants that have .npy files in encoded_root."""
    avail = []
    for v in VARIANT_NAMES:
        d = os.path.join(encoded_root, v)
        if os.path.exists(d) and any(f.endswith('.npy') for f in os.listdir(d)):
            avail.append(v)
    return avail


# ── Quick dataset check ───────────────────────────────────────────────────────
print("\nChecking available ablation variants...")
try:
    avail = available_variants(ENCODED_AMSTE)
    print(f"  Found {len(avail)}/{len(VARIANT_NAMES)}: "
          f"{[VARIANT_DISPLAY[v] for v in avail]}")
    missing = [v for v in VARIANT_NAMES if v not in avail]
    if missing:
        print(f"    Missing: {missing} — re-run notebook 3 to generate them.")
except FileNotFoundError:
    print(f"    ENCODED_AMSTE not found: {ENCODED_AMSTE}")
    print(f"  Run notebook 3.0 first to generate the ablation variants.")


# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — SNN Model Architecture                                        ║
# ╚══════════════════════════════════════════════════════════════════════════╝

class DAS_SNN(nn.Module):
    """
    Feedforward SNN: Dropout(0.1) → FC(n→128) → LIF → FC(128→64) → LIF → FC(64→2) → LIF

    β hierarchy (all learnable via surrogate gradient):
        β1=0.85 — layer 1: fast decay, detects sharp P-wave transients
        β2=0.90 — layer 2: medium integration
        β3=0.95 — output:  slow accumulation over the full 1-second window

    Input dropout (p=0.1) is applied per timestep before fc1. It prevents
    co-adaptation during the initial training plateau common with sparse
    encoders (Basic, No-Polarity), and is disabled automatically during
    evaluation (model.eval()).

    Weight init: Kaiming uniform for all FC layers (explicit, not default).

    SynOps formula for FC(n→128→64→2):
        SynOps = (input_spikes × 128) + (L1_spikes × 64) + (L2_spikes × 2)
    Dropout is not applied during eval, so SynOps reflects true inference cost.
    """
    def __init__(self, num_inputs=NUM_INPUTS_AMSTE, num_hidden1=NUM_HIDDEN1,
                  num_hidden2=NUM_HIDDEN2, num_outputs=NUM_OUTPUTS,
                  num_steps=NUM_STEPS):
        super().__init__()
        self.num_steps  = num_steps
        self.num_inputs = num_inputs
        spike_grad = surrogate.fast_sigmoid(slope=SLOPE)

        # Input dropout — reduces co-adaptation during the initial plateau and
        # stabilises training for sparse ablation variants (Basic, No-Polarity).
        self.drop_in = nn.Dropout(p=0.1)

        self.fc1  = nn.Linear(num_inputs, num_hidden1)
        self.lif1 = snn.Leaky(beta=BETA1, learn_beta=True,
                               spike_grad=spike_grad, threshold=THRESHOLD,
                               reset_mechanism='subtract')
        self.fc2  = nn.Linear(num_hidden1, num_hidden2)
        self.lif2 = snn.Leaky(beta=BETA2, learn_beta=True,
                               spike_grad=spike_grad, threshold=THRESHOLD,
                               reset_mechanism='subtract')
        self.fc3  = nn.Linear(num_hidden2, num_outputs)
        self.lif3 = snn.Leaky(beta=BETA3, learn_beta=True,
                               spike_grad=spike_grad, threshold=THRESHOLD,
                               reset_mechanism='subtract')

        # Explicit Kaiming initialisation for all linear layers
        for m in [self.fc1, self.fc2, self.fc3]:
            nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
            nn.init.zeros_(m.bias)

    def forward(self, x):
        """
        x: (num_steps, batch, num_inputs)
        Returns spk_rec (num_steps, batch, num_outputs),
                mem_rec (num_steps, batch, num_outputs).
        """
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        mem3 = self.lif3.init_leaky()
        spk_rec, mem_rec = [], []

        for step in range(self.num_steps):
            inp  = self.drop_in(x[step])
            cur1 = self.fc1(inp);  spk1, mem1 = self.lif1(cur1, mem1)
            cur2 = self.fc2(spk1); spk2, mem2 = self.lif2(cur2, mem2)
            cur3 = self.fc3(spk2); spk3, mem3 = self.lif3(cur3, mem3)
            spk_rec.append(spk3)
            mem_rec.append(mem3)

        return torch.stack(spk_rec), torch.stack(mem_rec)

    def forward_with_internals(self, x):
        """Full forward pass returning all intermediate spike + membrane tensors."""
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        mem3 = self.lif3.init_leaky()
        r = {k: [] for k in ['spk1','mem1','spk2','mem2','spk3','mem3']}

        for step in range(self.num_steps):
            inp  = self.drop_in(x[step])
            cur1 = self.fc1(inp);  s1, mem1 = self.lif1(cur1, mem1)
            cur2 = self.fc2(s1);   s2, mem2 = self.lif2(cur2, mem2)
            cur3 = self.fc3(s2);   s3, mem3 = self.lif3(cur3, mem3)
            r['spk1'].append(s1); r['mem1'].append(mem1)
            r['spk2'].append(s2); r['mem2'].append(mem2)
            r['spk3'].append(s3); r['mem3'].append(mem3)

        return {k: torch.stack(v) for k, v in r.items()}


def load_checkpoint(variant_name, mode, fold=0, seed=42):
    """
    Load saved model checkpoint.
    mode: 'ablation' | 'quicktest'
    """
    path = os.path.join(CKPT_DIR,
                        f'{variant_name}_{mode}_fold{fold}_seed{seed}.pt')
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Checkpoint not found: {path}\n"
            f"Run the corresponding training cell first.")
    state = torch.load(path, map_location='cpu')
    n_inp = state['fc1.weight'].shape[1]
    model = DAS_SNN(num_inputs=n_inp).to(DEVICE)
    model.load_state_dict(state)
    model.eval()
    return model


def compute_synops(model, x_input):
    """Compute SynOps + energy for one sample."""
    model.eval()
    with torch.no_grad():
        internals = model.forward_with_internals(x_input.to(DEVICE))
    n_h1 = model.fc1.out_features
    n_h2 = model.fc2.out_features
    n_out = model.fc3.out_features
    i_sp = x_input.sum().item()
    l1   = internals['spk1'].sum().item()
    l2   = internals['spk2'].sum().item()
    ops  = i_sp*n_h1 + l1*n_h2 + l2*n_out
    return {'synops': ops, 'msynops': ops/1e6,
            'energy_uj': ops * ENERGY_PER_SYNOP_PJ * 1e-6}


# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Training + Evaluation Functions                               ║
# ╚══════════════════════════════════════════════════════════════════════════╝

def _metrics(targets, preds, probs):
    """
    Full metric suite for binary event/noise classification.

    Primary (ablation paper):
        f1, recall, fnr, fpr, auprc
    Secondary:
        accuracy, balanced_acc, precision, specificity, auc, mcc
    """
    cm             = confusion_matrix(targets, preds)
    tn, fp, fn, tp = cm.ravel()
    return {
        # ── primary ──────────────────────────────────────────────
        'f1':           f1_score(targets, preds, zero_division=0),
        'recall':       recall_score(targets, preds, zero_division=0),
        'fnr':          fn/(fn+tp)   if (fn+tp)>0   else 0.0,
        'fpr':          fp/(fp+tn)   if (fp+tn)>0   else 0.0,
        'auprc':        average_precision_score(targets, probs),
        # ── secondary ────────────────────────────────────────────
        'accuracy':     accuracy_score(targets, preds),
        'balanced_acc': balanced_accuracy_score(targets, preds),
        'precision':    precision_score(targets, preds, zero_division=0),
        'specificity':  tn/(tn+fp)   if (tn+fp)>0   else 0.0,
        'auc':          roc_auc_score(targets, probs),
        'mcc':          matthews_corrcoef(targets, preds),
        # ── diagnostics ──────────────────────────────────────────
        'cm':           cm.tolist(),
    }


def _forward_batch(model, loader, loss_fn, train=False, optimizer=None):
    """One pass over loader. Returns metrics dict."""
    if train: model.train()
    else:     model.eval()

    total_loss, n = 0.0, 0
    preds_all, tgt_all, prob_all = [], [], []

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for data, targets in loader:
            data    = data.permute(1, 0, 2).to(DEVICE)
            targets = targets.to(DEVICE)
            spk_rec, _ = model(data)
            loss = loss_fn(spk_rec, targets)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item(); n += 1
            counts = spk_rec.sum(dim=0)
            probs  = torch.softmax(counts.float(), dim=1)
            preds_all.extend(counts.argmax(dim=1).detach().cpu().numpy())
            tgt_all.extend(targets.detach().cpu().numpy())
            prob_all.extend(probs[:, 1].detach().cpu().numpy())

    m = _metrics(np.array(tgt_all), np.array(preds_all), np.array(prob_all))
    m['loss'] = total_loss / max(n, 1)
    return m


def train_one_run(variant_name, file_ids, labels, train_idx, val_idx,
                   seed, fold, mode, num_inputs=NUM_INPUTS_AMSTE,
                   encoded_root=ENCODED_AMSTE):
    """
    Train one fold/seed combination.
    mode: string tag for checkpoint naming ('ablation' | 'quicktest').
    Returns best validation metrics dict + saves checkpoint.
    """
    torch.manual_seed(seed); np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    train_ds = DASSpikeDataset(file_ids[train_idx], labels[train_idx],
                                encoded_root, variant_name)
    val_ds   = DASSpikeDataset(file_ids[val_idx], labels[val_idx],
                                encoded_root, variant_name)
    tr_ld = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=0, pin_memory=False)
    va_ld = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0, pin_memory=False)

    model     = DAS_SNN(num_inputs=num_inputs).to(DEVICE)
    loss_fn   = SF.mse_count_loss(correct_rate=CORRECT_RATE,
                                   incorrect_rate=INCORRECT_RATE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=LR_FACTOR, patience=LR_PATIENCE)

    best_f1, best_meta, best_w = -1.0, None, None
    best_epoch, no_imp = 0, 0

    for epoch in range(NUM_EPOCHS):
        t0  = time.time()
        trn = _forward_batch(model, tr_ld, loss_fn, train=True,
                              optimizer=optimizer)
        val = _forward_batch(model, va_ld, loss_fn)
        scheduler.step(val['loss'])
        lr = optimizer.param_groups[0]['lr']

        # Reject degenerate all-positive solutions
        # (F1 ≈ 0.667 with FNR ≈ 0 on balanced data = predicting all events).
        is_degenerate = (val['f1'] <= 0.70 and val['fnr'] < 0.05)
        past_burnin   = (epoch + 1) >= BURNIN_EPOCHS

        if val['f1'] > best_f1 and not is_degenerate:
            best_f1 = val['f1']; best_meta = val.copy()
            best_w  = copy.deepcopy(model.state_dict())
            best_epoch = epoch+1; no_imp = 0
        elif past_burnin:
            no_imp += 1

        deg_flag  = ' [DEG]'  if is_degenerate  else ''
        burn_flag = ' [BURN]' if not past_burnin else ''
        print(f"  Ep {epoch+1:>3d}  "
              f"| TR  loss={trn['loss']:.4f} acc={trn['accuracy']:.3f} "
              f"f1={trn['f1']:.3f} auc={trn['auc']:.3f}  "
              f"| VA  loss={val['loss']:.4f} acc={val['accuracy']:.3f} "
              f"f1={val['f1']:.3f} auc={val['auc']:.3f}  "
              f"lr={lr:.2e}  "
              f"{'★' if no_imp==0 and past_burnin else ' '}"
              f"{deg_flag}{burn_flag}  ({time.time()-t0:.1f}s)")

        if past_burnin and no_imp >= EARLY_STOP_PATIENCE:
            print(f"  ⏹ Early stop @epoch {epoch+1}  "
                  f"(best F1={best_f1:.4f} @ep{best_epoch})")
            break

    if best_w is None:
        # All epochs degenerate — use final weights and warn.
        print(f"    WARNING: no non-degenerate checkpoint for "
              f"{variant_name} fold={fold} seed={seed}. Using final weights.")
        best_w     = copy.deepcopy(model.state_dict())
        best_meta  = val.copy()
        best_epoch = NUM_EPOCHS

    model.load_state_dict(best_w)
    best_meta.update({
        'variant': variant_name, 'fold': fold, 'seed': seed,
        'mode': mode, 'best_epoch': best_epoch,
        'betas': {'b1': model.lif1.beta.item(),
                  'b2': model.lif2.beta.item(),
                  'b3': model.lif3.beta.item()},
    })
    ckpt = os.path.join(CKPT_DIR,
                        f'{variant_name}_{mode}_fold{fold}_seed{seed}.pt')
    torch.save(best_w, ckpt)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return best_meta


def evaluate_loader(model, loader, loss_fn=None):
    """Evaluate model on a DataLoader, return metrics dict."""
    if loss_fn is None:
        loss_fn = SF.mse_count_loss(correct_rate=CORRECT_RATE,
                                     incorrect_rate=INCORRECT_RATE)
    return _forward_batch(model, loader, loss_fn)


def compute_efficiency_metrics(model, loader, num_inputs, num_steps=NUM_STEPS):
    """
    Compute spike-rate and SynOps efficiency metrics over a full DataLoader.

    Returns dict with:
        input_spike_rate  — fraction of input neurons firing per timestep
        l1_spike_rate     — same for hidden layer 1 (128 units)
        l2_spike_rate     — same for hidden layer 2 (64 units)
        synops            — mean total synaptic operations per sample
        msynops           — synops / 1e6
        energy_uj         — mean energy estimate (µJ) using Loihi-2 figure
        inference_time_ms — wall-clock time per sample

    SynOps = (input_spikes × 128) + (L1_spikes × 64) + (L2_spikes × 2)
    """
    model.eval()
    n_h1, n_h2, n_out = (model.fc1.out_features, model.fc2.out_features,
                          model.fc3.out_features)
    inp_rates, l1_rates, l2_rates, synops_all = [], [], [], []
    t_total = 0.0

    with torch.no_grad():
        for data, _ in loader:
            data = data.permute(1, 0, 2).to(DEVICE)
            B    = data.shape[1]
            t0   = time.time()
            internals = model.forward_with_internals(data)
            t_total  += time.time() - t0

            inp_spikes = data.sum(dim=0).sum(dim=1)
            l1_spikes  = internals['spk1'].sum(dim=0).sum(dim=1)
            l2_spikes  = internals['spk2'].sum(dim=0).sum(dim=1)

            inp_rates.extend((inp_spikes / (num_steps * num_inputs)).cpu().tolist())
            l1_rates.extend( (l1_spikes  / (num_steps * n_h1)).cpu().tolist())
            l2_rates.extend( (l2_spikes  / (num_steps * n_h2)).cpu().tolist())

            s = inp_spikes * n_h1 + l1_spikes * n_h2 + l2_spikes * n_out
            synops_all.extend(s.cpu().tolist())

    n_samples   = len(inp_rates)
    mean_synops = float(np.mean(synops_all))
    return {
        'input_spike_rate':  float(np.mean(inp_rates)),
        'l1_spike_rate':     float(np.mean(l1_rates)),
        'l2_spike_rate':     float(np.mean(l2_rates)),
        'synops':            mean_synops,
        'msynops':           mean_synops / 1e6,
        'energy_uj':         mean_synops * ENERGY_PER_SYNOP_PJ * 1e-6,
        'inference_time_ms': (t_total / max(n_samples, 1)) * 1e3,
    }


def save_results(rows, filename):
    """Append rows (list of dicts) to a CSV in RESULTS_DIR."""
    path = os.path.join(RESULTS_DIR, filename)
    new_df = pd.DataFrame(rows)
    if os.path.exists(path):
        new_df = pd.concat([pd.read_csv(path), new_df], ignore_index=True)
    new_df.to_csv(path, index=False)
    print(f"  Results → {path}")


# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 5 — AMSTE ABLATION: Within-dataset 5-fold CV                     ║
# ║                                                                          ║
# ║  Trains the same SNN on each of the 6 ablation variants using            ║
# ║  identical splits/folds/seeds/hyperparameters.                           ║
# ║                                                                          ║
# ║  Design:                                                                  ║
# ║    1. Fixed 15% held-out test set (~600 files) — same for all variants.  ║
# ║    2. Remaining 85% (~3400 files) → 5-fold stratified CV.                ║
# ║    3. 3 seeds per fold → 15 runs per variant → mean±std.                 ║
# ║    4. Best CV checkpoint → evaluate once on held-out test set.           ║
# ║                                                                          ║
# ║  Run one variant:   ablation_single('Full_AMSTE')                       ║
# ║  Run all variants:  ablation_all()                                       ║
# ║  Evaluate test set: ablation_test_eval('Full_AMSTE')                    ║
# ╚══════════════════════════════════════════════════════════════════════════╝

def ablation_get_split(encoded_root=ENCODED_AMSTE):
    """Fixed train/val pool + test split. Returns arrays."""
    file_ids, labels, _ = load_labels(encoded_root, 'amste')
    tv_ids, te_ids, tv_lab, te_lab = train_test_split(
        file_ids, labels, test_size=TEST_SIZE,
        stratify=labels, random_state=SPLIT_SEED)
    print(f"\nAblation split (seed={SPLIT_SEED}):")
    print(f"  Train/Val: {len(tv_ids):>5d}  "
          f"({np.sum(tv_lab==1)} ev + {np.sum(tv_lab==0)} no)")
    print(f"  Test:      {len(te_ids):>5d}  "
          f"({np.sum(te_lab==1)} ev + {np.sum(te_lab==0)} no)")
    return tv_ids, tv_lab, te_ids, te_lab


def ablation_single(variant_name, n_folds=NUM_FOLDS, seeds=SEEDS,
                     encoded_root=ENCODED_AMSTE):
    """
    Train one ablation variant with stratified CV.
    Returns list of per-fold/seed result dicts.
    """
    if variant_name not in VARIANT_NAMES:
        raise ValueError(f"Unknown variant: {variant_name}. "
                         f"Valid: {VARIANT_NAMES}")
    n_inp = infer_num_inputs(encoded_root, variant_name)
    tv_ids, tv_lab, _, _ = ablation_get_split(encoded_root)
    disp = VARIANT_DISPLAY[variant_name]

    print(f"\n{'='*70}")
    print(f"AMSTE ABLATION — Within-dataset CV: {disp}")
    print(f"  n_inputs={n_inp} | {n_folds} folds × {len(seeds)} seeds")
    print(f"{'='*70}")

    skf     = StratifiedKFold(n_splits=n_folds, shuffle=True,
                              random_state=SPLIT_SEED)
    results = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(tv_ids, tv_lab)):
        for seed in seeds:
            print(f"\n  ── Fold {fold+1}/{n_folds}  Seed {seed} ──")
            r = train_one_run(
                variant_name, tv_ids, tv_lab, tr_idx, va_idx,
                seed=seed, fold=fold, mode='ablation',
                num_inputs=n_inp, encoded_root=encoded_root)
            results.append(r)
            print(f"     Best val F1={r['f1']:.3f}  "
                  f"AUC={r['auc']:.3f}  FNR={r['fnr']:.3f}")

    # Print fold summary
    df = pd.DataFrame(results)
    print(f"\n  WITHIN-CV SUMMARY ({disp}):")
    primary   = ['f1', 'recall', 'fnr', 'fpr', 'auprc']
    secondary = ['accuracy', 'balanced_acc', 'precision', 'specificity',
                 'auc', 'mcc']
    print(f"  {'─'*50}")
    print(f"  Primary metrics:")
    for metric in primary:
        if metric in df.columns:
            print(f"    {metric:14s}: {df[metric].mean():.4f} ± "
                  f"{df[metric].std():.4f}")
    print(f"  Secondary metrics:")
    for metric in secondary:
        if metric in df.columns:
            print(f"    {metric:14s}: {df[metric].mean():.4f} ± "
                  f"{df[metric].std():.4f}")

    save_results(results, 'ablation_within_cv_results.csv')
    return results


def ablation_test_eval(variant_name, encoded_root=ENCODED_AMSTE):
    """
    Evaluate the best ablation checkpoint on the held-out test set.
    'Best' = fold/seed with highest val F1 from ablation_within_cv_results.csv.
    """
    csv_path = os.path.join(RESULTS_DIR, 'ablation_within_cv_results.csv')
    if not os.path.exists(csv_path):
        raise FileNotFoundError("Run ablation_single() first.")
    df  = pd.read_csv(csv_path)
    sub = df[df['variant'] == variant_name]
    if len(sub) == 0:
        raise ValueError(f"No ablation results for {variant_name}")
    best = sub.loc[sub['f1'].idxmax()]
    fold, seed = int(best['fold']), int(best['seed'])
    disp = VARIANT_DISPLAY[variant_name]
    print(f"\nAblation test eval — {disp}  "
          f"(best val F1={best['f1']:.3f}, fold={fold}, seed={seed})")

    _, _, te_ids, te_lab = ablation_get_split(encoded_root)
    n_inp = infer_num_inputs(encoded_root, variant_name)
    model = load_checkpoint(variant_name, 'ablation', fold=fold, seed=seed)
    te_ds = DASSpikeDataset(te_ids, te_lab, encoded_root, variant_name)
    te_ld = DataLoader(te_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0)
    m = evaluate_loader(model, te_ld)

    # Efficiency metrics on test set
    print(f"  Computing efficiency metrics on test set...")
    eff = compute_efficiency_metrics(model, te_ld,
                                     num_inputs=n_inp, num_steps=NUM_STEPS)
    m.update(eff)
    m.update({'variant': variant_name, 'mode': 'ablation_test',
              'fold': fold, 'seed': seed, 'n_test': len(te_ids)})
    _print_test_metrics(m, f"AMSTE Ablation Test Set — {disp}")
    save_results([m], 'ablation_test_results.csv')
    return m


def ablation_all(n_folds=NUM_FOLDS, seeds=SEEDS,
                  encoded_root=ENCODED_AMSTE, run_test_eval=True):
    """
    Run the full ablation: train all 6 variants, then test-eval each.
    Expected runtime: ~12–18 hrs on a single Jetson AGX Orin GPU
    (≈ 6 variants × 15 runs × ~ 8 min each).
    """
    avail = available_variants(encoded_root)
    print(f"\nAMSTE ABLATION — Running {len(avail)}/{len(VARIANT_NAMES)} "
          f"variants on June_24")
    print(f"  Variants in order: {[VARIANT_DISPLAY[v] for v in avail]}")
    all_results = []
    for v in avail:
        try:
            r = ablation_single(v, n_folds=n_folds, seeds=seeds,
                                 encoded_root=encoded_root)
            all_results.extend(r)
            if run_test_eval:
                try:
                    ablation_test_eval(v, encoded_root=encoded_root)
                except Exception as ex:
                    print(f"   test_eval {v}: {ex}")
        except Exception as ex:
            print(f"  {v}: {ex}")
    return all_results


# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 6 — Summary + Ablation Tables                                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝

def _print_test_metrics(m, title):
    """Pretty-print full classification + efficiency metrics for one variant."""
    cm = np.array(m['cm'])
    tn, fp, fn, tp = cm.ravel()
    w = 60
    print(f"\n{'─'*w}")
    print(f"  {title}")
    print(f"{'─'*w}")
    print(f"  ── Classification (primary) ─────────────────────────")
    print(f"  F1 Score      : {m['f1']:.4f}")
    print(f"  Recall        : {m['recall']:.4f}  "
          f"({tp} detected / {fn+tp} events)")
    print(f"  FNR           : {m['fnr']:.4f}  "
          f"({fn} missed / {fn+tp} events)")
    print(f"  FPR           : {m['fpr']:.4f}  "
          f"({fp} false alarms / {tn+fp} noise)")
    print(f"  AUPRC         : {m.get('auprc', float('nan')):.4f}")
    print(f"  ── Classification (secondary) ───────────────────────")
    print(f"  Accuracy      : {m['accuracy']:.4f}")
    print(f"  Balanced Acc  : {m.get('balanced_acc', float('nan')):.4f}")
    print(f"  Precision     : {m['precision']:.4f}")
    print(f"  Specificity   : {m.get('specificity', float('nan')):.4f}")
    print(f"  AUC-ROC       : {m['auc']:.4f}")
    print(f"  MCC           : {m.get('mcc', float('nan')):.4f}")
    if 'input_spike_rate' in m:
        print(f"  ── Efficiency ───────────────────────────────────────")
        print(f"  Input spike rate : {m['input_spike_rate']:.4f}  "
              f"(L1: {m['l1_spike_rate']:.4f}  "
              f"L2: {m['l2_spike_rate']:.4f})")
        print(f"  SynOps           : {m['msynops']:.4f} MSynOps  "
              f"({m['synops']:.0f} total)")
        print(f"  Energy estimate  : {m['energy_uj']:.4f} µJ  "
              f"(Loihi-2 @ {ENERGY_PER_SYNOP_PJ} pJ/SynOp)")
        print(f"  Inference time   : {m.get('inference_time_ms', 0):.3f} "
              f"ms/sample")
    print(f"  ── Confusion Matrix ─────────────────────────────────")
    print(f"              Pred Noise  Pred Event")
    print(f"  True Noise    {tn:>6d}      {fp:>6d}")
    print(f"  True Event    {fn:>6d}      {tp:>6d}")


def print_ablation_summary():
    """
    Print three tables:
      Table 1 — Classification metrics (mean±std over CV runs)
      Table 2 — Efficiency metrics (best checkpoint per variant)
      Table 3 — Leave-one-out Δ vs Full AMSTE (paper headline table)
    """
    cv_path   = os.path.join(RESULTS_DIR, 'ablation_within_cv_results.csv')
    test_path = os.path.join(RESULTS_DIR, 'ablation_test_results.csv')

    # ── Table 1: Classification (within-CV, mean ± std) ───────────────────
    if not os.path.exists(cv_path):
        print("No CV results — run ablation_all() first.")
        return
    df_cv = pd.read_csv(cv_path)
    best_f1 = df_cv.groupby('variant')['f1'].mean().max()
    print(f"\n{'='*108}")
    print(f"  TABLE 1 — Classification Metrics  "
          f"(AMSTE Ablation Within-CV, mean ± std over "
          f"{NUM_FOLDS * len(SEEDS)} runs)")
    print(f"{'='*108}")
    hdr = (f"{'Variant':>14}  {'F1':>13}  {'Recall':>13}  "
           f"{'FNR':>13}  {'FPR':>13}  {'AUPRC':>13}  "
           f"{'AUC':>13}  {'MCC':>8}")
    print(hdr)
    print('─' * len(hdr))
    for v in VARIANT_NAMES:
        sub = df_cv[df_cv['variant'] == v]
        if len(sub) == 0: continue
        disp = VARIANT_DISPLAY[v]
        flag = ' ◀' if abs(sub['f1'].mean() - best_f1) < 1e-9 else ''
        def ms(col):
            return (f"{sub[col].mean():.4f}±{sub[col].std():.3f}"
                    if col in sub.columns else '  N/A      ')
        print(f"  {disp:>12}  {ms('f1'):>13}  {ms('recall'):>13}  "
              f"{ms('fnr'):>13}  {ms('fpr'):>13}  {ms('auprc'):>13}  "
              f"{ms('auc'):>13}  "
              f"{sub['mcc'].mean():>7.4f}{flag}")

    # ── Table 2: Efficiency (test set, best checkpoint) ───────────────────
    if not os.path.exists(test_path):
        print(f"\nNo test results — run ablation_test_eval() or "
              f"ablation_all(run_test_eval=True).")
        return
    df_te = pd.read_csv(test_path)
    print(f"\n{'='*98}")
    print(f"  TABLE 2 — Efficiency Metrics  (Test Set, best CV checkpoint)")
    print(f"{'='*98}")
    hdr2 = (f"{'Variant':>14}  {'F1':>8}  {'InSpRate':>10}  "
            f"{'L1SpRate':>10}  {'MSynOps':>10}  "
            f"{'Energy(µJ)':>12}  {'InfTime(ms)':>12}")
    print(hdr2)
    print('─' * len(hdr2))
    for v in VARIANT_NAMES:
        sub = df_te[df_te['variant'] == v]
        if len(sub) == 0: continue
        disp = VARIANT_DISPLAY[v]
        def g(col):
            return f"{sub[col].mean():.4f}" if col in sub.columns else 'N/A'
        print(f"  {disp:>12}  {g('f1'):>8}  "
              f"{g('input_spike_rate'):>10}  {g('l1_spike_rate'):>10}  "
              f"{g('msynops'):>10}  {g('energy_uj'):>12}  "
              f"{g('inference_time_ms'):>12}")

    # ── Table 3: Leave-one-out Δ vs Full AMSTE (paper headline) ───────────
    full_cv = df_cv[df_cv['variant'] == 'Full_AMSTE']
    if len(full_cv) == 0:
        print(f"\n   Full_AMSTE results missing — cannot build Δ table.")
        return
    full_te = df_te[df_te['variant'] == 'Full_AMSTE']
    fmean   = full_cv.mean(numeric_only=True)
    f_eff   = full_te.mean(numeric_only=True) if len(full_te) else None

    print(f"\n{'='*120}")
    print(f"  TABLE 3 — Leave-One-Out Ablation: Δ vs Full AMSTE "
          f"(paper headline table)")
    print(f"{'='*120}")
    print(f"  {'Variant':14} | {'MAD':>3} | {'Pol':>3} | {'Multi':>5} | "
          f"{'Spat':>4} | {'F1':>14} | {'Recall':>14} | {'FNR':>14} | "
          f"{'AUPRC':>14} | {'Energy(µJ)':>10}")
    print("  " + "-" * 114)
    yn = lambda b: '✔' if b else '✘'
    for v in VARIANT_NAMES:
        sub    = df_cv[df_cv['variant'] == v]
        sub_te = df_te[df_te['variant'] == v]
        if len(sub) == 0: continue
        disp = VARIANT_DISPLAY[v]
        m, p, ms_, sg = VARIANT_FLAGS[v]
        # Δ vs Full AMSTE; for Full itself, show absolute values not deltas
        if v == 'Full_AMSTE':
            f1_str    = f"{sub['f1'].mean():.4f}"
            rec_str   = f"{sub['recall'].mean():.4f}"
            fnr_str   = f"{sub['fnr'].mean():.4f}"
            au_str    = f"{sub['auprc'].mean():.4f}"
            marker    = ' FULL'
        else:
            df1  = sub['f1'].mean()    - fmean['f1']
            drec = sub['recall'].mean() - fmean['recall']
            dfnr = sub['fnr'].mean()   - fmean['fnr']
            daup = sub['auprc'].mean() - fmean['auprc']
            sign = lambda x: f"+{x:.4f}" if x >= 0 else f"{x:.4f}"
            f1_str  = f"{sub['f1'].mean():.4f} ({sign(df1)})"
            rec_str = f"{sub['recall'].mean():.4f} ({sign(drec)})"
            fnr_str = f"{sub['fnr'].mean():.4f} ({sign(dfnr)})"
            au_str  = f"{sub['auprc'].mean():.4f} ({sign(daup)})"
            marker  = ''
        e_str = (f"{sub_te['energy_uj'].mean():.3f}"
                 if len(sub_te) and 'energy_uj' in sub_te.columns else 'N/A')
        print(f"  {disp:14} | {yn(m):>3} | {yn(p):>3} | {yn(ms_):>5} | "
              f"{yn(sg):>4} | {f1_str:>14} | {rec_str:>14} | {fnr_str:>14} | "
              f"{au_str:>14} | {e_str:>10}{marker}")

    print(f"\n  ✔ = component active     ✘ = component removed (ablated)")
    print(f"  Δ values shown vs Full AMSTE. Negative Δ on F1/Recall/AUPRC "
          f"means the component MATTERS.")
    print(f"  Positive Δ on FNR means removing that component INCREASES "
          f"missed events.")

    # Save the headline table as CSV for the paper
    headline_rows = []
    for v in VARIANT_NAMES:
        sub = df_cv[df_cv['variant'] == v]
        sub_te = df_te[df_te['variant'] == v]
        if len(sub) == 0: continue
        m, p, ms_, sg = VARIANT_FLAGS[v]
        row = {
            'variant': v, 'display': VARIANT_DISPLAY[v],
            'mad': m, 'polarity': p,
            'multi_scale': ms_, 'spatial_gate': sg,
            'f1_mean':       sub['f1'].mean(),
            'f1_std':        sub['f1'].std(),
            'recall_mean':   sub['recall'].mean(),
            'recall_std':    sub['recall'].std(),
            'fnr_mean':      sub['fnr'].mean(),
            'fnr_std':       sub['fnr'].std(),
            'auprc_mean':    sub['auprc'].mean(),
            'auprc_std':     sub['auprc'].std(),
            'auc_mean':      sub['auc'].mean(),
            'mcc_mean':      sub['mcc'].mean(),
        }
        if v != 'Full_AMSTE':
            row.update({
                'df1':    sub['f1'].mean()    - fmean['f1'],
                'drecall': sub['recall'].mean() - fmean['recall'],
                'dfnr':   sub['fnr'].mean()   - fmean['fnr'],
                'dauprc': sub['auprc'].mean() - fmean['auprc'],
            })
        if len(sub_te):
            row.update({
                'energy_uj_test': sub_te['energy_uj'].mean()
                                  if 'energy_uj' in sub_te.columns else None,
                'msynops_test':   sub_te['msynops'].mean()
                                  if 'msynops' in sub_te.columns else None,
                'f1_test':        sub_te['f1'].mean(),
            })
        headline_rows.append(row)
    out_csv = os.path.join(RESULTS_DIR, 'ablation_headline_table.csv')
    pd.DataFrame(headline_rows).to_csv(out_csv, index=False)
    print(f"\n  Headline table → {out_csv}")


def plot_ablation_results(metric='f1'):
    """
    Bar chart of AMSTE ablation within-CV results per variant with error bars.
    Full AMSTE has a black border (proposed method).
    Basic has a dashed border (lower-bound baseline).
    """
    p = os.path.join(RESULTS_DIR, 'ablation_within_cv_results.csv')
    if not os.path.exists(p):
        print("No results found. Run ablation_all() first.")
        return
    df    = pd.read_csv(p)
    stats = df.groupby('variant')[metric].agg(['mean', 'std'])
    variants = [v for v in VARIANT_NAMES if v in stats.index]
    means    = [stats.loc[v, 'mean'] for v in variants]
    stds     = [stats.loc[v, 'std']  for v in variants]
    colors   = [VARIANT_COLORS[v]    for v in variants]
    labels   = [VARIANT_DISPLAY[v]   for v in variants]

    fig, ax = plt.subplots(figsize=(13, 5))
    bars = ax.bar(range(len(variants)), means, yerr=stds, capsize=4,
                   color=colors, alpha=0.85, edgecolor='white', linewidth=0.5)
    for bar, mn, sd in zip(bars, means, stds):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + sd + 0.01,
                f'{mn:.3f}', ha='center', va='bottom', fontsize=8)

    # Highlight Full AMSTE (proposed) and Basic (lower bound)
    for i, v in enumerate(variants):
        if v == 'Full_AMSTE':
            bars[i].set_edgecolor('black')
            bars[i].set_linewidth(2.5)
        elif v == 'Basic':
            bars[i].set_edgecolor('dimgray')
            bars[i].set_linewidth(1.5)
            bars[i].set_linestyle('--')

    ax.set_xticks(range(len(variants)))
    ax.set_xticklabels(labels, rotation=25, ha='right', fontsize=10)
    ax.set_ylabel(metric.upper(), fontsize=11)
    if metric in ('f1', 'recall', 'auprc', 'auc', 'precision', 'accuracy'):
        ax.set_ylim(0, 1.12)
    ax.grid(axis='y', alpha=0.3)
    ax.set_title(f'AMSTE Ablation — {metric.upper()} per Variant\n'
                 f'(Black border = Full AMSTE proposed | Dashed = Basic lower-bound | '
                 f'{NUM_FOLDS} folds × {len(SEEDS)} seeds)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    out = os.path.join(RESULTS_DIR, f'ablation_{metric}_comparison.png')
    fig.savefig(out, dpi=200, bbox_inches='tight')
    plt.show()
    print(f"Saved: {out}")


def plot_efficiency_frontier():
    """
    Scatter plot: F1 (y) vs MSynOps (x) per variant.
    Ideal variants sit in the top-left (high F1, low compute).
    Full AMSTE marked with a black ring.
    """
    p = os.path.join(RESULTS_DIR, 'ablation_test_results.csv')
    if not os.path.exists(p):
        print("No test results — run ablation_test_eval() first.")
        return
    df = pd.read_csv(p)
    if 'msynops' not in df.columns:
        print("No efficiency metrics in test results.")
        return

    fig, ax = plt.subplots(figsize=(10, 6))
    for v in VARIANT_NAMES:
        sub = df[df['variant'] == v]
        if len(sub) == 0: continue
        x, y = sub['msynops'].mean(), sub['f1'].mean()
        c = VARIANT_COLORS[v]
        marker     = 'D' if VARIANT_TYPE[v] == 'lower_bound' else 'o'
        edge_color = 'black' if v == 'Full_AMSTE' else 'white'
        edge_width = 2.5    if v == 'Full_AMSTE' else 1.0
        ax.scatter(x, y, color=c, s=160, marker=marker, zorder=3,
                   edgecolors=edge_color, linewidth=edge_width)
        ax.annotate(VARIANT_DISPLAY[v], (x, y), textcoords='offset points',
                    xytext=(8, 4), fontsize=9, color=c,
                    fontweight='bold' if v == 'Full_AMSTE' else 'normal')

    ax.set_xlabel('MSynOps (× 10⁶ synaptic operations per inference)',
                  fontsize=11)
    ax.set_ylabel('F1 Score (test set)', fontsize=11)
    ax.set_title('AMSTE Ablation — Efficiency Frontier  '
                 '(F1 vs Computational Cost)\n'
                 '◆ Lower-bound  ● Leave-one-out / Full  |  '
                 'top-left = best trade-off',
                 fontsize=12, fontweight='bold')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    out = os.path.join(RESULTS_DIR, 'ablation_efficiency_frontier.png')
    fig.savefig(out, dpi=200, bbox_inches='tight')
    plt.show()
    print(f"Saved: {out}")


# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 7 — Quick Test + Usage Examples                                   ║
# ╚══════════════════════════════════════════════════════════════════════════╝

def quick_test(variant_name='Full_AMSTE', n_files=50,
                encoded_root=ENCODED_AMSTE):
    """
    Sanity check: train 1 fold × 1 seed on n_files for 3 epochs, print F1.
    Use before committing to a full run (~2 min on Jetson AGX Orin).
    """
    n_inp = infer_num_inputs(encoded_root, variant_name)
    file_ids, labels, _ = load_labels(encoded_root, 'test')

    rng  = np.random.default_rng(42)
    sel  = rng.choice(len(file_ids), min(n_files, len(file_ids)), replace=False)
    ids  = file_ids[sel]; lab = labels[sel]
    n_tr = int(0.8 * len(ids))
    tr_idx = np.arange(n_tr); va_idx = np.arange(n_tr, len(ids))

    global NUM_EPOCHS, EARLY_STOP_PATIENCE
    orig_ep = NUM_EPOCHS
    orig_es = EARLY_STOP_PATIENCE

    disp = VARIANT_DISPLAY[variant_name]
    print(f"\nQuick test: {disp} | {n_files} files | "
          f"3 epochs | n_inputs={n_inp}")

    NUM_EPOCHS          = 3
    EARLY_STOP_PATIENCE = 10

    r = train_one_run(variant_name, ids, lab, tr_idx, va_idx,
                       seed=42, fold=0, mode='quicktest',
                       num_inputs=n_inp, encoded_root=encoded_root)

    NUM_EPOCHS          = orig_ep
    EARLY_STOP_PATIENCE = orig_es
    print(f"\n  Quick test result: F1={r['f1']:.3f}  AUC={r['auc']:.3f}")
    print(f" Variant loads + trains correctly.")
    return r


"""
═══════════════════════════════════════════════════════════════════════════
USAGE GUIDE — AMSTE Ablation Study
═══════════════════════════════════════════════════════════════════════════

── 1. Sanity check (always run first, ~2 min) ──────────────────────────────
quick_test('Full_AMSTE', n_files=100)
quick_test('Basic',      n_files=100)   # confirm lower-bound trains too

── 2. Run the ablation: train all 6 variants + test-eval ──────────────────
# One variant at a time (recommended — monitor first):
results = ablation_single('Full_AMSTE')
results = ablation_single('Basic')
results = ablation_single('No_MAD')
results = ablation_single('No_Polarity')
results = ablation_single('No_MultiScale')
results = ablation_single('No_Spatial')

# Or all 6 at once (long — ~12-18 hrs total):
all_results = ablation_all()

# Evaluate best checkpoint on held-out test set:
for v in ['Full_AMSTE', 'Basic', 'No_MAD',
          'No_Polarity', 'No_MultiScale', 'No_Spatial']:
    ablation_test_eval(v)

── 3. Summary + plots (run after ablation_all()) ───────────────────────────
print_ablation_summary()                # Tables 1 (CV) + 2 (test eff) + 3 (Δ headline)
plot_ablation_results(metric='f1')
plot_ablation_results(metric='recall')
plot_ablation_results(metric='fnr')
plot_ablation_results(metric='auprc')
plot_efficiency_frontier()              # F1 vs MSynOps scatter

── 4. Paper wording template (after results are in) ────────────────────────
'''
To examine the contribution of each AMSTE component, an ablation study was
performed on the FRISCO FORGE 2024 dataset. The ablation removed one of the
four AMSTE components at a time — adaptive MAD thresholding, polarity
separation, multi-scale temporal voting, or the spatial coherence gate —
while keeping the SNN architecture, training procedure, data split, and
hyperparameters unchanged. A Basic change encoder, in which all four
components are removed, was included as the lower-bound baseline.

Each variant was trained with 5-fold stratified cross-validation and three
random seeds (15 runs per variant) on 85 % of the dataset, with the
remaining 15 % held out as a fixed test set used for the final efficiency
and classification metrics. Differences between variants in F1, recall,
false-negative rate, AUPRC, and energy consumption can therefore be
attributed solely to the ablated component.
'''
═══════════════════════════════════════════════════════════════════════════
"""

# ── Default run: full ablation + summary + plots ────────────────────────────
# Comment out the next two blocks for an interactive session.
all_results = ablation_all()

print_ablation_summary()
plot_ablation_results(metric='f1')
plot_ablation_results(metric='recall')
plot_ablation_results(metric='fnr')
plot_ablation_results(metric='auprc')
plot_efficiency_frontier()
